# Lab 59: The Default Mode

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 7, *The Decision Bias*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-59.ipynb)

**What you will do:** compare your own opt-in vs opt-out survey results with the published organ-donation figures, pool the class's data, and run a toy recommender simulation to see how a single early default can snowball into a population-wide habit.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

If you extend this Lab to a topic closer to the original organ-donation research rather than the
book's suggested "email updates" example, keep the survey neutral and hypothetical. Do not use it
to probe anyone's real, personal medical decisions.

## The experiment in the book

In Lab 59 you tested the **default effect**: you sent one group a survey with a box
**pre-checked** (they had to actively uncheck it to decline) and another group the same survey
with the box **unchecked** (they had to actively check it to agree). Johnson and Goldstein
(2003) found that opt-out organ-donation policies produced registered-consent rates above 90%,
compared with under 20% under opt-in policies — the same decision, framed only by which box was
already ticked. A caveat from the book: registering as a consenting donor is not the same as
actually becoming one, and later research found that opt-out policy alone does not reliably
predict how many organs are actually transplanted.

## Record your results

Enter the number of people who agreed out of those surveyed, for each version.

In [ ]:
results = pd.DataFrame({
    "version": ["A: opt-out (pre-checked)", "B: opt-in (unchecked)"],
    "agreed": [9, 3],     # example data: replace with your own counts
    "surveyed": [10, 10], # example data
})
results["agree_rate"] = results.agreed / results.surveyed
display(results)
results.plot.bar(x="version", y="agree_rate", legend=False, color=["steelblue", "grey"], figsize=(5.5, 3))
plt.ylabel("proportion agreeing"); plt.ylim(0, 1); plt.xticks(rotation=15, ha="right"); plt.show()

## Compare

The lab reports opt-out consent rates **above 90%** against opt-in rates **below 20%** for
organ donation policy, comparing whole countries rather than a small survey. Treat this as
context for the *direction and rough scale* of the default effect, not a target your own
small sample must hit.

In [ ]:
opt_out = results.loc[results.version.str.contains("opt-out"), "agree_rate"].iloc[0]
opt_in = results.loc[results.version.str.contains("opt-in"), "agree_rate"].iloc[0]
print(f"Your opt-out agreement rate: {opt_out:.0%}   |   published range: above 90%")
print(f"Your opt-in agreement rate:  {opt_in:.0%}   |   published range: below 20%")
if opt_out > opt_in:
    print("Your survey shows the same direction as the published effect: the default pulled")
    print("people towards it. A small survey will rarely match the exact percentages of a")
    print("national policy comparison, so focus on the direction and the size of the gap.")
else:
    print("Your survey did not show the predicted direction. With a small sample this can")
    print("happen by chance; a genuine reversal would be a striking and worth rechecking.")

## The AI side

The book connects the default effect to the **cold-start problem** in recommender systems: a
new user has no history, so the system falls back on a default (often "whatever is currently
popular"), and the user's resulting clicks feed back into what counts as popular. Below, a toy
population of simulated users each pick from a small set of items, weighted partly by their
own taste and partly by whichever item the system currently recommends as the default. Compare
a system that keeps recommending the early leader against one that recommends a random item
each time.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
rng = np.random.default_rng(2026)
n_items, n_users, default_weight = 5, 300, 0.6

def simulate(strategy, rng):
    counts = np.ones(n_items)  # start with a weak, uninformative prior
    history = []
    for _ in range(n_users):
        taste = rng.dirichlet(np.ones(n_items))  # this user's private preference
        if strategy == "sticky_default":
            default_item = np.argmax(counts)
            probs = np.ones(n_items) * 0.01
            probs[default_item] += default_weight
            probs = (1 - default_weight) * taste + default_weight * (probs / probs.sum())
        else:  # random_default: no reinforcement, a fresh random suggestion each time
            default_item = rng.integers(n_items)
            probs = (1 - default_weight) * taste + default_weight * np.eye(n_items)[default_item]
        probs = probs / probs.sum()
        choice = rng.choice(n_items, p=probs)
        counts[choice] += 1
        history.append(choice)
    return counts / counts.sum(), history

sticky_share, _ = simulate("sticky_default", rng)
random_share, _ = simulate("random_default", rng)
share_df = pd.DataFrame({"sticky (popularity) default": sticky_share, "random default": random_share})
display(share_df.round(2))
share_df.plot.bar(figsize=(6, 3)); plt.ylabel("share of all choices"); plt.xlabel("item"); plt.show()
print(f"Most popular item's share under the sticky default: {sticky_share.max():.0%}")
print(f"Most popular item's share under the random default: {random_share.max():.0%}")

A system that keeps recommending "whatever is already popular" locks in an early leader, even
though every simulated user has an equally random private taste — the default, not the taste,
decides the outcome. A system that resets to a random suggestion each time never lets one item
dominate. This is a toy model, not a real recommender: real systems have far richer signals,
but the basic mechanism — the default shapes the data, and the data reinforces the default —
is the same one the book describes for the cold-start problem.

## Pool the class data

Pool the class's opt-in/opt-out agreement counts (anonymous IDs, not names).

In [ ]:
import io
csv_text = """id,version,agreed,surveyed
P01,opt-out,8,10
P01,opt-in,2,10
P02,opt-out,9,10
P02,opt-in,4,10
P03,opt-out,10,10
P03,opt-in,1,10
"""  # example data: replace with your class CSV, e.g. pd.read_csv("default_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
summary = df.groupby("version")[["agreed", "surveyed"]].sum()
summary["rate"] = summary.agreed / summary.surveyed
display(summary)
boot = {v: [df.loc[df.version == v].sample(frac=1, replace=True).agreed.sum()
            / df.loc[df.version == v].sample(frac=1, replace=True).surveyed.sum()
            for _ in range(3000)] for v in ["opt-out", "opt-in"]}
gap = np.array(boot["opt-out"]) - np.array(boot["opt-in"])
lo, hi = np.percentile(gap, [2.5, 97.5])
print(f"Class pooled gap (opt-out minus opt-in): {gap.mean():.0%} (95% bootstrap interval {lo:.0%}-{hi:.0%})")
print("Published gap for comparison: roughly 70+ points (above 90% vs below 20%)")

## Questions to think about

1. Did your own opt-in and opt-out rates move in the same direction as the published organ-donation figures, even if the exact percentages were very different?
2. The book stresses that registered consent is not the same as actual donation. What would you need to measure, beyond agreement rates, to know whether a default actually changes real-world outcomes rather than just stated intentions?
3. In the recommender simulation, what happens to the sticky default's dominant item if you lower default_weight towards 0? What does that represent about how strongly a real system leans on its default?
4. The book calls unchosen hyperparameters in machine-learning code "the pre-checked boxes of AI design." Can you think of a default setting in an app or website you use that shapes behaviour the way the pre-checked donation box does?

## Challenge

Design a third survey version that is genuinely neutral — no box pre-checked either way, just a
plain yes/no question with no default. Run it alongside your opt-in and opt-out versions and see
where the neutral condition falls.

In [ ]:
# Example data: replace with your own three-way comparison.
three_way = pd.DataFrame({
    "version": ["opt-out", "neutral (no default)", "opt-in"],
    "agree_rate": [0.85, 0.45, 0.25],   # example data
})
display(three_way)
three_way.plot.bar(x="version", y="agree_rate", legend=False, color="steelblue", figsize=(5.5, 3))
plt.ylabel("proportion agreeing"); plt.ylim(0, 1); plt.xticks(rotation=15, ha="right"); plt.show()
print("If the neutral condition falls roughly between opt-out and opt-in, that supports the idea")
print("that both defaults are actively pulling behaviour away from a baseline, not just that one")
print("of them is 'normal'. The numbers above are example data -- replace them with your own.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-59.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")